## 컴퓨터 비전(Computer Vision) 관련 기술 실습

### CNN(Convolution Neural Networks)

#### Convolution : 합성곱 연산 

In [ ]:
import numpy as np

# input array
m = np.array([[1,2,2,0],
              [0,1,2,3],
              [1,0,1,2],
              [2,3,0,1]])

# convolution kernel(==filter,mask)
f = np.array([[0,0,0],
              [0,1,0],
              [0,0,0]])

# output
result = []

mx, my = np.shape(m)
fx, fy = np.shape(f)
print(f'mx, my : {mx, my}')
print(f'fx, fy : {fx, fy}')

for i in range(mx-fx+1):
    for j in range(my-fy+1):
        result.append((m[i:i+fy, j:j+fy] * f).sum())
        
result=np.array(result).reshape(2,2)        
print(result)

### 영상 처리 예제

#### 라이브러리 설치하기 : OpenCV 

In [ ]:
!pip install opencv-python

In [ ]:
import cv2
cv2.__version__
help(cv2)

- 영상(이미지) 부드럽게 / 날카롭게

In [ ]:
import sys
import numpy as np
import cv2

src = cv2.imread('image/lenna.bmp', cv2.IMREAD_GRAYSCALE)

if src is None:
    print('Image load failed!')
    sys.exit()

# 부드럽게
# mask = np.array([[1/9,1/9,1/9],
#                 [1/9,1/9,1/9],
#                 [1/9,1/9,1/9]])
# mask = np.ones((3,3), dtype=np.float64)/ 9.

# 날카롭게
mask = np.array([[0,-1,0],
                [-1,4,-1],
                [0,-1,0]])

dst = cv2.filter2D(src, -1, mask)

cv2.imshow('src', src)
cv2.imshow('dst', dst)
cv2.waitKey()  # 키가 눌려질때까지 기다리기
cv2.destroyAllWindows()

- 에지 검출

In [ ]:
# Canny 에지 검출
src = cv2.imread('image/lenna.bmp', cv2.IMREAD_GRAYSCALE)

dst = cv2.Canny(src, 50, 150)

cv2.imshow('src', src)
cv2.imshow('dst', dst)
cv2.waitKey()
cv2.destroyAllWindows()

- 영상 부드럽게(blurring)

In [ ]:
import sys
import numpy as np
import cv2

src = cv2.imread('image/lenna.bmp', cv2.IMREAD_GRAYSCALE)

if src is None:
    print('Image load failed!')
    sys.exit()

cv2.imshow('src', src)

for sigma in range(1, 6):
    # sigma 값을 이용하여 가우시안 필터링
    dst = cv2.GaussianBlur(src, (0, 0), sigma)

    desc = 'sigma = {}'.format(sigma)
    cv2.putText(dst, desc, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                1.0, 255, 1, cv2.LINE_AA)

    cv2.imshow('dst', dst)
    cv2.waitKey()

cv2.destroyAllWindows()

- 영상 날카롭게(Sharpening)

In [ ]:
import sys
import numpy as np
import cv2

src = cv2.imread('image/lenna.bmp', cv2.IMREAD_GRAYSCALE)

if src is None:
    print('Image load failed!')
    sys.exit()

cv2.imshow('src', src)

src_f = src.astype(np.float32)
for sigma in range(1, 6):
    blr = cv2.GaussianBlur(src_f, (0, 0), sigma)
    dst = np.clip(2.0*src_f - blr, 0, 255).astype(np.uint8)
    
    desc = 'sigma = {}'.format(sigma)
    cv2.putText(dst, desc, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                1.0, 255, 1, cv2.LINE_AA)
    
    cv2.imshow('dst', dst)
    cv2.waitKey()

cv2.destroyAllWindows()

- 노이즈 영상 화질 개선

In [ ]:
import sys
import numpy as np
import cv2

src = cv2.imread('image/lenna_noise.bmp', cv2.IMREAD_GRAYSCALE)

if src is None:
    print('Image load failed!')
    sys.exit()

dst = cv2.medianBlur(src, 5)

cv2.imshow('src', src)
cv2.imshow('dst', dst)
cv2.waitKey()

cv2.destroyAllWindows()

------------------------------------

### [실습] 실시간 카툰 필터 카메라 만들기

- 실시간 카메라 실행

In [ ]:
import sys
import cv2

# 카메라 열기
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Camera open failed!")
    sys.exit()

# 카메라 프레임 처리
while True:
    ret, frame = cap.read()
    if not ret: 
        print("frame read failed!")
        sys.exit()

    cv2.imshow('frame', frame)

    if cv2.waitKey(10) == 27:  # ESC키 눌려지면
        break

cap.release()
cv2.destroyAllWindows()

- 실시간 카툰 필터 카메라 만들기

In [ ]:
import sys
import cv2

def cartoon_filter(img):
    h, w = img.shape[:2]
    img2 = cv2.resize(img, (w//2, h//2))

    blr = cv2.bilateralFilter(img2, -1, 20, 7)
    edge = 255 - cv2.Canny(img2, 80, 120)
    edge = cv2.cvtColor(edge, cv2.COLOR_GRAY2BGR)

    dst = cv2.bitwise_and(blr, edge)
    dst = cv2.resize(dst, (w, h), interpolation=cv2.INTER_NEAREST)
    return dst


def pencil_sketch_filter(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blr = cv2.GaussianBlur(gray, (0, 0), 3)
    dst = cv2.divide(gray, blr, scale=255)
    return dst

# 실시간 카메라 작동
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print('Video open failed!')
    sys.exit()

cam_mode = 0    
while True:
    ret, frame = cap.read()
    if not ret:
        print('Video read failed!')
        break
            
    # 모드에 따라 필터 적용한 영상(frame) 보여주기
    if cam_mode == 1:
        frame = cartoon_filter(frame)
        title = f'{cam_mode} : cartoon_filter'
        tcolor = (255,0,255)
    elif cam_mode == 2:
        frame = pencil_sketch_filter(frame)
        title = f'{cam_mode} : pencil_sketch_filter'
        tcolor = (0,0,0)
    else:
        title = f'{cam_mode} :  '
        tcolor = (0,0,0)
    
    cv2.putText(frame, title, (30, 30),
            cv2.FONT_HERSHEY_DUPLEX, 1, 
             tcolor, 2, cv2.LINE_AA)
    cv2.imshow('frame', frame)
    

    key = cv2.waitKey(1)
    if key==27: # ESC키 눌러졌을 때
        break
    elif key==ord(' '): #스페이스키 눌러졌을 때
        cam_mode += 1        
        print(f'cam_mode:{cam_mode}')
        if cam_mode==3:
            cam_mode = 0
    
cap.release()
cv2.destroyAllWindows()
print('Video Finish!')

--------------------

## Mediapipe 

- 구글이 만든 컴퓨터 비전용 머신러닝 라이브러리
- 참고: https://google.github.io/mediapipe

#### 라이브러리 설치하기

In [ ]:
!pip install mediapipe

#### 라이브러리 사용하기

In [ ]:
import mediapipe as mp

In [ ]:
help(mp)

### [실습] Mediapipe를 이용한 얼굴 검출

https://google.github.io/mediapipe/solutions/face_detection.html

In [ ]:
import cv2
import mediapipe as mp

mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

# For webcam input:
cap = cv2.VideoCapture(0)

with mp_face_detection.FaceDetection(
    model_selection=0, min_detection_confidence=0.5) as face_detection:
    
    while cap.isOpened():
        success, image = cap.read()
        if not success:
            print("Ignoring empty camera frame.")
            # If loading a video, use 'break' instead of 'continue'.
            continue

        # To improve performance, optionally mark the image as not writeable to
        # pass by reference.
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_detection.process(image)

        # Draw the face detection annotations on the image.
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        if results.detections:
            for detection in results.detections:
                mp_drawing.draw_detection(image, detection)
                #print(detection)
                
        # Flip the image horizontally for a selfie-view display.
        cv2.imshow('MediaPipe Face Detection', cv2.flip(image, 1))
        if cv2.waitKey(5) & 0xFF == 27:
            break
            
cap.release()
cv2.destroyAllWindows()

### [실습] Mediapipe Selfie Segmentation
- https://google.github.io/mediapipe/solutions/selfie_segmentation 

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
mp_drawing = mp.solutions.drawing_utils
mp_selfie_segmentation = mp.solutions.selfie_segmentation

# For webcam input:
BG_COLOR = (192, 192, 192) # gray
cap = cv2.VideoCapture(0)
with mp_selfie_segmentation.SelfieSegmentation(
    model_selection=1) as selfie_segmentation:
    bg_image = None
    while cap.isOpened():
        success, image = cap.read()
        if not success:
            print("Ignoring empty camera frame.")
            # If loading a video, use 'break' instead of 'continue'.
            continue

        # Flip the image horizontally for a later selfie-view display, and convert
        # the BGR image to RGB.
        image = cv2.cvtColor(cv2.flip(image, 1), cv2.COLOR_BGR2RGB)
        # To improve performance, optionally mark the image as not writeable to
        # pass by reference.
        image.flags.writeable = False
        results = selfie_segmentation.process(image)

        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Draw selfie segmentation on the background image.
        # To improve segmentation around boundaries, consider applying a joint
        # bilateral filter to "results.segmentation_mask" with "image".
        condition = np.stack(
                  (results.segmentation_mask,) * 3, axis=-1) > 0.1
        # The background can be customized.
        #   a) Load an image (with the same width and height of the input image) to
        #      be the background, e.g., bg_image = cv2.imread('/path/to/image/file')
        #   b) Blur the input image by applying image filtering, e.g.,
        #      bg_image = cv2.GaussianBlur(image,(55,55),0)
        if bg_image is None:
            bg_image = np.zeros(image.shape, dtype=np.uint8)
            bg_image[:] = BG_COLOR
            
        output_image = np.where(condition, image, bg_image)

        cv2.imshow('MediaPipe Selfie Segmentation', output_image)
        if cv2.waitKey(5) & 0xFF == 27:
            break

    cap.release()

----------------------

In [ ]:
끝